In [41]:
import coiled

import fsspec
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging
from flox.xarray import xarray_reduce
import numpy as np
import dask
import re
import sparse
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy

# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

In [2]:
logging.getLogger("distributed.client").setLevel(logging.ERROR)

In [3]:
fs = fsspec.filesystem("s3", requester_pays=True)

In [ ]:
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=2,
    tags={"project": "AFOLU_flux_model"},
    scheduler_vm_types="r5.2xlarge", # memory optimized AWS EC2 instances
    worker_vm_types="r5.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

In [ ]:
client.restart() 

In [4]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 7
Total threads: 14,Total memory: 31.08 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35019,Workers: 7
Dashboard: http://127.0.0.1:8787/status,Total threads: 14
Started: Just now,Total memory: 31.08 GiB
Comm: tcp://127.0.0.1:46647,Total threads: 2
Dashboard: http://127.0.0.1:44499/status,Memory: 4.44 GiB
Nanny: tcp://127.0.0.1:37089,


In [ ]:
local_client.shutdown()

In [112]:
# Test of explicit uri listing

model_version = "version_0_3_2"
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
run_date = "20250507"
interval = "2015_2016"

gross_emis_CO2_2016_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_{interval}.tif"], 
                                         name = 'emis_all_C_pools_CO2_only')
gross_emis_all_gases_2016_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/{interval}/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_{interval}.tif"], 
                                         name = 'emis_all_C_pools_all_gases')
nodes_2016_tile_uri = pd.Series([f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__land_state_node_{interval}.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__land_state_node_{interval}.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__land_state_node_{interval}.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/{interval}/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__land_state_node_{interval}.tif"], 
                          name = 'node_codes')
print(gross_emis_CO2_2016_tile_uri)
# print(gross_emis_all_gases_2016_tile_uri)
# print(nodes_2016_tile_uri)
# print(type(gross_emis_CO2_2016_tile_uri))

0    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
1    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
2    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
3    s3://gfw2-data/climate/AFOLU_flux_model/LULUCF...
Name: emis_all_C_pools_CO2_only, dtype: object


In [113]:
# uri components

model_version = "version_0_3_2"
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"
run_date = "20250507"
interval_end_years = [2016]

In [126]:
# Node codes output from model. Covers entire decision tree. Make sure that node codes are right-padded with 0s to 7 digits! 
# Otherwise, only the node codes that are seven digits without 0s will be matched with the node code rasters and output. 
# TODO: I may have accidentally missed some node codes when copying them from the decision tree. Check!
node_codes = np.array([
    1110000, 1120000, 1210000, 1220000, 2111000, 2112000,
    2121100, 2121200, 2122100, 2122200, 2123100, 2123200,
    2124100, 2124200, 2125100, 2125200, 2211100, 2211200, 2212110, 2212120, 
    2212210, 2212220, 2213110, 2213120, 2213210, 2213220,
    2214100, 2214200, 2215100, 2215200, 2221100, 2221200, 2223100, 2223200,
    2222100, 2222200, 3110000, 3120000, 3211211, 3211212,
    3211221, 3211222, 3212111, 3212112, 3212121, 3212122,
    3212211, 3212212, 3212221, 3212222, 3221110, 3221120,
    3221210, 3221220, 3222111, 3222112, 3222121, 3222122,
    3222210, 3222220, 4100000, 4210000, 4220000, 4310000,
    4320000, 5100000, 5210000, 5220000, 5310000, 5320000],
dtype=np.uint32)

# # Node codes output from model for 2x2 test area in DRC (23_-5_25_-3)
# node_codes = np.array([3222111, 2212120, 3222121, 3212122, 2212110, 3212121, 3222210, 2223200, 
#                        2221200, 3212222, 2221100, 5220000, 4100000, 2223100,
#                        2212220, 3212221, 2211200, 5210000, 2211100, 4220000, 2212210, 2214100, 2214200, 4210000, 2215200], 
#                       dtype=np.uint32)

In [120]:
# Extracts some metadata/chunk properties to add to the output dataframe
def parse_metadata_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__(\d+_-?\d+_\d+_-?\d+)__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_pixel_yr_(\d{4}_\d{4})\.tif$"
    match = re.search(pattern, uri)

    if match:
        chunk_id = match.group(1)
        variable = match.group(2)
        interval = match.group(3)
    else:
        interval, chunk_id, variable = None, None, None

    return interval, chunk_id, variable

In [7]:
def make_xarray_chunks(tile_uris):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True
        # chunks={'x': 1000, 'y':1000}
    ).squeeze().persist()

    return xarray_chunks

In [8]:
def align_with_nodes(analysis_layer, nodes):
    analysis_layer_sub, nodes_aligned = xr.align(analysis_layer, nodes, join="inner")
    return analysis_layer_sub, nodes_aligned

In [13]:
def xarray_reduction(analysis_layer, node_data):

    analysis_layer_by_node = xarray_reduce(
        analysis_layer.band_data,
        node_data,
        func='sum',
        expected_groups=(node_codes),
        reindex=ReindexStrategy(
            blockwise=False, array_type=ReindexArrayType.SPARSE_COO
        ),
        fill_value=0   
    )

    return analysis_layer_by_node

In [31]:
def create_output_year_df(output_year_result, interval, ouput_pattern):

    output_year_result_sparse_data = output_year_result.data

    # Step 3: Extract coordinates and values
    dim_names = output_year_result.dims
    indices = output_year_result_sparse_data.coords
    output_year_values = output_year_result_sparse_data.data

    # Step 4: Map dimension indices to coordinate values
    output_year_coord_dict = {
        dim: output_year_result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    output_year_coord_dict["value"] = output_year_values
    
    output_year_coord_dict = {
        dim: output_year_result.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    output_year_coord_dict["value"] = output_year_values
    
    output_year_df = pd.DataFrame(output_year_coord_dict)

    output_year_df["interval_end"] = interval
    output_year_df["output_pattern"] = ouput_pattern

    return output_year_df

In [121]:
# uris for inputs, with placeholders for interval that are filled in as intervals are iterated through

gross_emis_CO2_2016_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_INTERVAL.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_INTERVAL.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_INTERVAL.tif",
                                          f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__gross_emissions__all_C_pools__CO2_only__MgCO2_pixel_yr_INTERVAL.tif"], 
                                         name = 'emis_all_C_pools_CO2_only')
gross_emis_all_gases_2016_tile_uri = pd.Series([f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_INTERVAL.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_INTERVAL.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_INTERVAL.tif",
                                           f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/annual_intervals/INTERVAL/_pixel_yr/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__gross_emissions__all_C_pools__all_gases__MgCO2e_pixel_yr_INTERVAL.tif"], 
                                         name = 'emis_all_C_pools_all_gases')
nodes_2016_tile_uri = pd.Series([f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/00N_020E__23_-4_24_-3__land_state_node_INTERVAL.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/00N_020E__23_-5_24_-4__land_state_node_INTERVAL.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/00N_020E__24_-4_25_-3__land_state_node_INTERVAL.tif",
                           f"{output_path}land_state_node/standard_model/annual_intervals/INTERVAL/4000_pixels/{run_date}/00N_020E__24_-5_25_-4__land_state_node_INTERVAL.tif"], 
                          name = 'node_codes')

In [127]:
analysis_layer_years = [gross_emis_CO2_2016_tile_uri, gross_emis_all_gases_2016_tile_uri]

combined_df = pd.DataFrame()

for analysis_layer_year in analysis_layer_years:

    for interval_end_year in interval_end_years:

        interval = f"{interval_end_year-1}_{interval_end_year}"
        # print(interval)

        # analysis_layer_year = [path.replace("INTERVAL", interval) for path in analysis_layer_year]
        analysis_layer_year = analysis_layer_year.str.replace("INTERVAL", interval, regex=False)
        # print(analysis_layer_year)

        # nodes_2016_tile_uri = [path.replace("INTERVAL", interval) for path in nodes_2016_tile_uri]
        nodes_2016_tile_uri = nodes_2016_tile_uri.str.replace("INTERVAL", interval, regex=False)

        interval_from_inputs, chunk_id, output_pattern = parse_metadata_from_uri(analysis_layer_year)
        # print(output_pattern)
    
        print(f"Processing {(output_pattern)} for {interval}")
    
        layer_xarray_chunks = make_xarray_chunks(analysis_layer_year)
        nodes_xarray_chunks = make_xarray_chunks(nodes_2016_tile_uri)
        # print("layer_xarray_chunks:", layer_xarray_chunks)
        # print("nodes_xarray_chunks:", nodes_xarray_chunks)
        
        layer_sub, nodes_aligned = align_with_nodes(layer_xarray_chunks, nodes_xarray_chunks)
        # print("layer_sub:", layer_sub)
        # print("nodes_aligned:", nodes_aligned)
        
        node_data = nodes_aligned.band_data
        node_data.name = 'state_node'
        # print("node_data", node_data)
        
        analysis_layer_by_node = xarray_reduction(layer_sub, node_data)
        # print("analysis_layer_by_node:", analysis_layer_by_node)
        
        analysis_layer_result = analysis_layer_by_node.compute()
        # print(analysis_layer_result)
        
        output_year_df = create_output_year_df(analysis_layer_result, interval_end_year, output_pattern)
        # print(output_year_df)
    
        combined_df = pd.concat([combined_df, output_year_df])

combined_df

Processing gross_emissions__all_C_pools__CO2_only__MgCO2 for 2015_2016
Processing gross_emissions__all_C_pools__all_gases__MgCO2e for 2015_2016


,state_node,value,interval_end,output_pattern
0,2211100,1.060868e+04,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
1,2211200,2.899860e+04,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
2,2212110,9.996288e+06,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
3,2212120,5.400350e+07,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
4,2212210,4.059757e+04,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
5,2212220,9.205915e+05,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
6,2214100,2.111563e+03,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
7,2214200,3.164155e+03,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
8,2215200,4.834308e+02,2016,gross_emissions__all_C_pools__CO2_only__MgCO2
9,2221100,5.077360e+04,2016,gross_emissions__all_C_pools__CO2_only__MgCO2


In [ ]:
combined_df.head()

In [ ]:
combined_df[(combined_df.state_node == 2212210)]